# S1 J4 — Pipeline bout en bout + FastAPI

**Objectif :** assembler toutes les briques validées individuellement (J1 à J3) en une seule chaîne
fonctionnelle : `ffmpeg → transcribe.py (ASR) → NLU (Rasa) → retriever.py (RAG) → LLM (génération) →
synthesize.py (TTS)`, exposée via FastAPI et branchée sur le canal Telegram déjà configuré.

**Fonctions déjà prêtes, à assembler :**
- `asr/transcribe.py` — `transcribe(wav_path, lang) → texte` (J1)
- `rag/retriever.py` — `retrieve(question, lang, mode) → passages` (J2)
- Génération LLM + `build_prompt()` (J3, à formaliser en fonction réutilisable)
- `tts/synthesize.py` — `synthesize(texte, lang) → audio` (J3)
- `app/channels/telegram.py` — réception/envoi de messages (S0/S1)

**Reste à faire :**
- Écrire `app/pipeline.py` : `process(audio_in) → audio_out`, orchestrant les briques ci-dessus
- Brancher le NLU (service Rasa via HTTP) — actuellement les briques testées isolément n'appellent pas encore Rasa
- Connecter `process_message()` dans `app/main.py` à ce vrai pipeline (aujourd'hui, accusé de réception simple)
- Journaliser chaque étape (texte transcrit, intent, fiches récupérées, réponse) pour le débogage
- Mesurer la latence de bout en bout


In [ ]:
import os
from pathlib import Path

try:
    import google.colab
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    PROJECT_ROOT = Path("/content/noo-far-pipeline")
    if not PROJECT_ROOT.exists():
        !git clone https://github.com/noofar-ia/noo-far-pipeline.git /content/noo-far-pipeline
    else:
        !cd {PROJECT_ROOT} && git pull
else:
    PROJECT_ROOT = Path(r"C:\dev\noo-far-pipeline")

import sys
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Projet : {PROJECT_ROOT} | Sur Colab : {ON_COLAB}")

In [ ]:
import requests
r = requests.post(
    "https://laughing-couscous-xr5gwp9w6p99h9qr6-5005.app.github.dev/",
    json={"text": "quand vacciner mes vaches"}
)
print(r.status_code)
print(r.json())

In [ ]:
import os
os.environ["RASA_URL_OVERRIDE"] = "https://laughing-couscous-xr5gwp9w6p99h9qr6-5005.app.github.dev"

In [ ]:
!sed -i 's|http://localhost:5005|https://laughing-couscous-xr5gwp9w6p99h9qr6-5005.app.github.dev|' {PROJECT_ROOT}/config/config.yaml

In [ ]:
!grep rasa_url {PROJECT_ROOT}/config/config.yaml

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from nlu.client import parse_intent
print(parse_intent("quand vacciner mes vaches"))

In [ ]:
!pip install chromadb rank-bm25 --quiet

In [ ]:
import chromadb
from rank_bm25 import BM25Okapi
print("OK")

In [ ]:
!cd {PROJECT_ROOT} && python rag/indexer.py

In [ ]:
from huggingface_hub import login
login(token="MOn_Token")  # remplace par ton vrai token, ne jamais committer avec la vraie valeur

In [ ]:
from rag.retriever import retrieve
passages = retrieve("Quand vacciner mes vaches ?", mode="semantic")
for doc, meta in passages:
    print(meta["source"], "-", doc[:100])

In [ ]:
from rag.generator import generate

question = "Quand vacciner mes vaches ?"
passages = retrieve(question, mode="semantic")
reponse = generate(question, passages)
print(reponse)

In [ ]:
from tts.synthesize import synthesize
from IPython.display import Audio, display

audio, sr = synthesize(reponse, lang="fr")
display(Audio(audio, rate=sr))

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT))

from app.gradio_app import demo

demo.launch(share=True)

In [ ]:
import pandas as pd

resultats_j6 = []